# TP 9 — Kafka → Structured Streaming → Iceberg### Module 5 — Traitement temps réel**Durée :** 2 heures · **Noté sur 20**---## Ce que vous devez savoir faire à la fin1. Brancher Structured Streaming sur Kafka et écrire dans une table Iceberg.2. Agréger sur une **fenêtre de temps événement**, avec watermark.3. **Observer** l'effet du watermark sur des retardataires injectés volontairement.4. Tuer le job et vérifier que la reprise repart au bon endroit.5. **Mesurer** les petits fichiers produits, et en tirer les conséquences.## Barème| Exercice | Sujet | Points ||---|---|---|| 1 | Brancher la chaîne | 4 || 2 | Fenêtres et temps événement | 4 || 3 | Le watermark et les retardataires | 6 || 4 | Tuer et reprendre | 3 || 5 | Les petits fichiers | 3 |

---# Exercice 1 — Brancher la chaîne  *(4 points)*

In [ ]:
# 1.1 — Session Spark avec Kafka et Icebergfrom pyspark.sql import SparkSession, functions as Ffrom pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampTypeimport json, time, threadingfrom datetime import datetime, timedeltafrom kafka import KafkaProducerfrom kafka.admin import KafkaAdminClient, NewTopicUTILISATEUR = "etudiant"KAFKA = "kafka:9092"TOPIC = "flux-transactions"spark = (SparkSession.builder    .appName("TP9 - chaine complete")    .master("local[4]")    .config("spark.sql.extensions",            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")    .config("spark.sql.catalog.lakehouse.type", "hadoop")    .config("spark.sql.catalog.lakehouse.warehouse", f"s3a://lakehouse/{UTILISATEUR}-tp9")    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")    .config("spark.hadoop.fs.s3a.path.style.access", "true")    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")    .config("spark.sql.shuffle.partitions", "4")    .getOrCreate())spark.sparkContext.setLogLevel("WARN")print("Spark", spark.version)

In [ ]:
# 1.2 — Le topic et un producteur en arrière-planadmin = KafkaAdminClient(bootstrap_servers=KAFKA)try:    admin.delete_topics([TOPIC]); time.sleep(3)except Exception:    passadmin.create_topics([NewTopic(name=TOPIC, num_partitions=4, replication_factor=1)])time.sleep(2)PAYS = ["FR", "DE", "ES", "IT", "BE"]_arret = threading.Event()def produire(duree_s=120, par_seconde=50, retard_proportion=0.0, retard_minutes=30):    """Produit des transactions. Une proportion peut etre datee dans le PASSE."""    import random    p = KafkaProducer(bootstrap_servers=KAFKA,                      key_serializer=lambda k: k.encode(),                      value_serializer=lambda v: json.dumps(v).encode())    debut, i = time.time(), 0    while time.time() - debut < duree_s and not _arret.is_set():        for _ in range(par_seconde):            maintenant = datetime.utcnow()            if random.random() < retard_proportion:                ts = maintenant - timedelta(minutes=retard_minutes)   # RETARDATAIRE            else:                ts = maintenant            pays = random.choice(PAYS)            p.send(TOPIC, key=pays, value={                "id_transaction": f"T{i:09d}",                "horodatage": ts.isoformat(),                "pays": pays,                "montant": round(random.lognormvariate(3.5, 0.9), 2)})            i += 1        p.flush()        time.sleep(1)    p.close()    print(f"producteur arrete apres {i} messages")print("fonction de production prete")

In [ ]:
# 1.3 — La table Iceberg de destinationspark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.flux")spark.sql("DROP TABLE IF EXISTS lakehouse.flux.par_pays_15min")spark.sql("""CREATE TABLE lakehouse.flux.par_pays_15min (    debut_fenetre TIMESTAMP,    fin_fenetre   TIMESTAMP,    pays          STRING,    nb            BIGINT,    total         DOUBLE) USING icebergPARTITIONED BY (days(debut_fenetre))""")print("table creee")

In [ ]:
# 1.4 — Lire Kafka en continu, et regarder ce qui arriveSCHEMA = StructType([    StructField("id_transaction", StringType()),    StructField("horodatage",     StringType()),    StructField("pays",           StringType()),    StructField("montant",        DoubleType()),])brut = (spark.readStream.format("kafka")        .option("kafka.bootstrap.servers", KAFKA)        .option("subscribe", TOPIC)        .option("startingOffsets", "earliest")        .load())evenements = (brut    .select(F.from_json(F.col("value").cast("string"), SCHEMA).alias("d"),            F.col("partition"), F.col("offset"))    .select("d.*", "partition", "offset")    .withColumn("horodatage", F.to_timestamp("horodatage")))print("colonnes brutes de Kafka :")brut.printSchema()

### Q1 *(4 pts)* —- **a.** Le schéma brut de Kafka contient sept colonnes. Lesquelles, et à quoi servent  `key`, `value`, `topic`, `partition`, `offset`, `timestamp` ?- **b.** Pourquoi faut-il un `from_json` ? Que contient `value` exactement ?- **c.** Le producteur utilise `pays` comme clé de partitionnement. Quelle conséquence sur la  répartition ? Est-ce un bon choix ici ?- **d.** `startingOffsets` vaut `earliest`. Quelle est l'autre valeur possible, et dans quel  cas la choisir ?

**Votre réponse :***(rédigez ici)*

---# Exercice 2 — Fenêtres et temps événement  *(4 points)*

In [ ]:
# 2.1 — Lancer le producteur en arrière-plan (2 minutes, sans retardataire)_arret.clear()th = threading.Thread(target=produire, kwargs=dict(duree_s=120, par_seconde=50,                                                   retard_proportion=0.0), daemon=True)th.start()print("producteur lance")time.sleep(10)

In [ ]:
# 2.2 — Agréger par fenêtre de temps ÉVÉNEMENTagrege = (evenements    .withWatermark("horodatage", "2 minutes")    .groupBy(F.window("horodatage", "1 minute").alias("f"), "pays")    .agg(F.count("*").alias("nb"), F.sum("montant").alias("total"))    .select(F.col("f.start").alias("debut_fenetre"),            F.col("f.end").alias("fin_fenetre"),            "pays", "nb", "total"))requete = (agrege.writeStream           .format("memory").queryName("apercu")           .outputMode("update")           .trigger(processingTime="10 seconds")           .start())print("requete lancee — attendez 40 s puis executez la cellule suivante")

In [ ]:
# 2.3 — Observertime.sleep(40)spark.sql("SELECT * FROM apercu ORDER BY debut_fenetre DESC, pays").show(20, truncate=False)print("progression :", json.dumps(requete.lastProgress, indent=2)[:900])

### Q2 *(4 pts)* —- **a.** Combien de fenêtres apparaissent ? Sont-elles closes ou encore en cours d'alimentation ?- **b.** Dans `lastProgress`, relevez `numInputRows`, `inputRowsPerSecond` et  `batchDuration`. Le traitement suit-il le rythme de production ?- **c.** On a utilisé le mode `update`. Que se passerait-il en mode `append` ? Pourquoi  n'aurait-on encore **rien** vu ?- **d.** Le `groupBy` porte sur `window("horodatage", ...)`. Que se passerait-il si l'on  utilisait à la place la colonne `timestamp` fournie par Kafka ?

**Votre réponse :***(rédigez ici)*

---# Exercice 3 — Le watermark et les retardataires  *(6 points)***Objectif.** Le cœur du module : mesurer ce qu'un watermark fait perdre.

## 3.1 — **PRÉDICTION** *(1 pt)*On va relancer le producteur en injectant **20 % d'événements datés de 30 minutes dans lepassé**, avec un watermark de **2 minutes**.Ces retardataires seront-ils comptés ? Dans quelle fenêtre ? Que verra-t-on dans lesmétriques ?

**Votre réponse :***(rédigez ici)*

In [ ]:
# 3.2 — Arrêter, puis relancer avec des retardatairesrequete.stop()_arret.set(); time.sleep(2); _arret.clear()th2 = threading.Thread(target=produire,                       kwargs=dict(duree_s=90, par_seconde=50,                                   retard_proportion=0.20, retard_minutes=30),                       daemon=True)th2.start()print("producteur relance avec 20 % de retardataires (-30 min)")time.sleep(5)

In [ ]:
# 3.3 — Même agrégation, watermark courtagrege_court = (evenements    .withWatermark("horodatage", "2 minutes")    .groupBy(F.window("horodatage", "1 minute").alias("f"), "pays")    .agg(F.count("*").alias("nb"))    .select(F.col("f.start").alias("debut"), "pays", "nb"))q_court = (agrege_court.writeStream.format("memory").queryName("court")           .outputMode("update").trigger(processingTime="10 seconds").start())time.sleep(45)print("=== fenetres vues avec un watermark de 2 minutes ===")spark.sql("SELECT debut, sum(nb) AS total FROM court GROUP BY debut ORDER BY debut").show(30, False)

In [ ]:
# 3.4 — La métrique qui compte : les lignes rejetéesprog = q_court.lastProgressfor op in prog.get("stateOperators", []):    print("numRowsTotal            :", op.get("numRowsTotal"))    print("numRowsUpdated          :", op.get("numRowsUpdated"))    print("numRowsDroppedByWatermark :", op.get("numRowsDroppedByWatermark"))print("\nwatermark courant :", prog.get("eventTime", {}).get("watermark"))print("min / max event time :", prog.get("eventTime", {}).get("min"),      "/", prog.get("eventTime", {}).get("max"))

In [ ]:
# 3.5 — Comparer avec un watermark LONGq_court.stop()agrege_long = (evenements    .withWatermark("horodatage", "45 minutes")    .groupBy(F.window("horodatage", "1 minute").alias("f"), "pays")    .agg(F.count("*").alias("nb"))    .select(F.col("f.start").alias("debut"), "pays", "nb"))q_long = (agrege_long.writeStream.format("memory").queryName("long")          .outputMode("update").trigger(processingTime="10 seconds").start())time.sleep(45)print("=== fenetres vues avec un watermark de 45 minutes ===")spark.sql("SELECT debut, sum(nb) AS total FROM long GROUP BY debut ORDER BY debut").show(40, False)prog = q_long.lastProgressfor op in prog.get("stateOperators", []):    print("numRowsTotal (etat conserve) :", op.get("numRowsTotal"))    print("numRowsDroppedByWatermark    :", op.get("numRowsDroppedByWatermark"))

### Q3 *(5 pts)* —- **a.** Avec le watermark court, combien de lignes ont été rejetées ? Cela correspond-il aux  20 % injectés ?- **b.** Avec le watermark long, les fenêtres d'il y a 30 minutes apparaissent-elles ?  Combien de lignes rejetées ?- **c.** Comparez `numRowsTotal` — la taille de l'état conservé — entre les deux. Quel rapport ?- **d.** Formulez l'arbitrage en une phrase, avec les deux coûts.- **e.** La perte du watermark court est-elle visible dans le résultat ? Comment un exploitant  pourrait-il s'en apercevoir en production ?

**Votre réponse :***(rédigez ici)*

---# Exercice 4 — Tuer et reprendre  *(3 points)*

In [ ]:
# 4.1 — Écrire pour de bon dans Iceberg, avec checkpointq_long.stop()CHECKPOINT = f"hdfs://namenode:8020/user/{UTILISATEUR}/checkpoints/tp9"final = (evenements    .withWatermark("horodatage", "2 minutes")    .groupBy(F.window("horodatage", "1 minute").alias("f"), "pays")    .agg(F.count("*").alias("nb"), F.sum("montant").alias("total"))    .select(F.col("f.start").alias("debut_fenetre"),            F.col("f.end").alias("fin_fenetre"),            "pays", "nb", "total"))q = (final.writeStream     .format("iceberg")     .outputMode("append")     .option("checkpointLocation", CHECKPOINT)     .option("fanout-enabled", "true")     .trigger(processingTime="15 seconds")     .toTable("lakehouse.flux.par_pays_15min"))print("ecriture Iceberg lancee — attendez 90 s")time.sleep(90)spark.table("lakehouse.flux.par_pays_15min").orderBy("debut_fenetre").show(20, False)

In [ ]:
# 4.2 — Noter la position, puis TUER le joboffsets_avant = q.lastProgress.get("sources", [{}])[0].get("endOffset")lignes_avant  = spark.table("lakehouse.flux.par_pays_15min").count()print("offsets avant arret :", offsets_avant)print("lignes dans la table :", lignes_avant)q.stop()print("\n>>> requete ARRETEE (simulation de panne)")time.sleep(5)

In [ ]:
# 4.3 — Redémarrer avec le MÊME checkpointq2 = (final.writeStream      .format("iceberg")      .outputMode("append")      .option("checkpointLocation", CHECKPOINT)      .option("fanout-enabled", "true")      .trigger(processingTime="15 seconds")      .toTable("lakehouse.flux.par_pays_15min"))time.sleep(30)offsets_apres = q2.lastProgress.get("sources", [{}])[0].get("startOffset")print("offsets au redemarrage :", offsets_apres)print("lignes dans la table   :", spark.table("lakehouse.flux.par_pays_15min").count())q2.stop()_arret.set()

### Q4 *(3 pts)* —- **a.** Comparez `endOffset` avant l'arrêt et `startOffset` au redémarrage. Que constatez-vous ?- **b.** Qu'est-ce qui, exactement, a permis cette reprise ? Que se serait-il passé sans  `checkpointLocation` ?- **c.** Vous modifiez la requête pour agréger par fenêtres de 5 minutes au lieu de 1, et vous  la redémarrez avec le même checkpoint. Que se passe-t-il ?

**Votre réponse :***(rédigez ici)*

---# Exercice 5 — Les petits fichiers  *(3 points)*

In [ ]:
# 5.1 — Combien de fichiers la chaîne a-t-elle produits ?stats = spark.sql("""SELECT count(*) AS fichiers,       sum(record_count) AS lignes,       round(avg(file_size_in_bytes)/1024, 1) AS taille_moyenne_kio,       round(sum(file_size_in_bytes)/1024/1024, 2) AS total_mioFROM lakehouse.flux.par_pays_15min.files""")stats.show()spark.sql("SELECT count(*) AS instantanes "          "FROM lakehouse.flux.par_pays_15min.snapshots").show()

In [ ]:
# 5.2 — Extrapolerres = stats.collect()[0]DUREE_TEST_MIN = 3          # duree approximative de production ci-dessuspar_minute = res.fichiers / DUREE_TEST_MINprint(f"fichiers produits en {DUREE_TEST_MIN} min : {res.fichiers}")print(f"soit environ {par_minute:.0f} fichiers/minute")print(f"\nSur 24 h  : {par_minute * 60 * 24:,.0f} fichiers")print(f"Sur 30 j  : {par_minute * 60 * 24 * 30:,.0f} fichiers")print(f"\nTaille moyenne : {res.taille_moyenne_kio} Kio")print("Recommandation du module 2 : 128 Mio a 1 Gio par fichier")

In [ ]:
# 5.3 — Compacterspark.sql("""CALL lakehouse.system.rewrite_data_files(    table => 'flux.par_pays_15min',    options => map('min-input-files', '2'))""").show(truncate=False)spark.sql("""SELECT count(*) AS fichiers,       round(avg(file_size_in_bytes)/1024, 1) AS taille_moyenne_kioFROM lakehouse.flux.par_pays_15min.files""").show()

### Q5 *(3 pts)* —- **a.** Combien de fichiers par jour cette chaîne produirait-elle ? Quelle taille moyenne ?- **b.** Comparez à la recommandation du module 2. De quel facteur êtes-vous en dessous ?- **c.** Proposez une politique complète pour cette chaîne : intervalle de déclenchement,  opérations de maintenance et fréquence. **Quel est le coût de votre proposition pour les  utilisateurs ?**

**Votre réponse :***(rédigez ici)*

---# Synthèse| Observation | Votre chiffre | Conséquence ||---|---|---|| Lignes rejetées, watermark 2 min | | || Lignes rejetées, watermark 45 min | | || État conservé, court vs long | | || Offsets avant / après redémarrage | | || Fichiers par jour extrapolés | | |**Question de conclusion.** Cette chaîne est-elle prête pour la production ? Citez les deuxchoses que vous mettriez en place avant de la déployer.

In [ ]:
# Nettoyagefor q in spark.streams.active:    q.stop()_arret.set()try:    admin.delete_topics([TOPIC])except Exception:    passspark.stop()print("Session fermee.")

---## Avant de rendre- [ ] La **prédiction** de 3.1 est écrite avant exécution.- [ ] Les questions **Q1 à Q5** sont rédigées et justifiées.- [ ] Les métriques `numRowsDroppedByWatermark` et `numRowsTotal` sont reportées.- [ ] Le tableau de synthèse est complété.- [ ] Notebook exporté en HTML et déposé.**Bon TP.**